# ETL — S&P 500 (mensual 2015–2025) con Refinitiv

**Objetivo:** construir un panel mensual (firm × month) a partir de datos 100% Refinitiv (Eikon/RDP) y exportar el dataset final para estimaciones econométricas.

**Inputs principales:**
- Universo de firmas: `empresas_tickers_test.csv`
- (Opcional) Excels de bonos corporativos: carpeta local (ver Config)

**Outputs principales:**
- `data/clean/panel_master.parquet`
- `data/clean/data_dictionary.csv`
- `outputs/qa/qa_report.csv` (o similar)

**Notas:**
- Requiere credenciales Refinitiv vía variable de entorno `EIKON_APP_KEY`.
- Algunas descargas pueden tardar y están sujetas a rate limits (HTTP 429). El notebook aplica pausas/reintentos donde corresponde.

## 2) Imports

In [ ]:
# --- Standard library
from __future__ import annotations

import os
import time
import logging
from pathlib import Path
from typing import Iterable, Optional

# --- Third-party
import numpy as np
import pandas as pd

# --- Refinitiv (Eikon)
import eikon as ek

## 3) Config

In [ ]:
# ======================
# CONFIG
# ======================

# ---- Horizonte temporal del panel (mensual 2015–2025)
START_DATE = "2015-01-01"
END_DATE   = "2025-12-31"

# ---- Root del proyecto (asume que corrés el notebook desde el repo)
PROJECT_ROOT = Path.cwd()

# ---- Estructura de datos dentro del repo
DATA_DIR   = PROJECT_ROOT / "data"
RAW_DIR    = DATA_DIR / "raw"
CLEAN_DIR  = DATA_DIR / "clean"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
QA_DIR     = OUTPUT_DIR / "qa"
LOG_DIR    = OUTPUT_DIR / "logs"

for p in (RAW_DIR, CLEAN_DIR, QA_DIR, LOG_DIR):
    p.mkdir(parents=True, exist_ok=True)

# ---- Inputs (ajustar a tus nombres reales)
UNIVERSE_FILE = DATA_DIR / "inputs" / "empresas_tickers_test.csv"

# Si hoy tenés excels de bonos en otra ruta, traelos al repo o apuntá acá de forma relativa
BONDS_DIR = DATA_DIR / "inputs" / "bonds_empresas"

# ---- Outputs (centralizá acá los nombres que YA estás usando)
OUT = {
    # universe/metadata
    "universe_firms": CLEAN_DIR / "universe_firms.parquet",
    "firms_metadata": CLEAN_DIR / "firms_metadata.parquet",
    "bonds_metadata": CLEAN_DIR / "bonds_metadata.parquet",

    # equity
    "equity_prices_daily": RAW_DIR / "equity_prices_daily.parquet",
    "equity_returns_monthly": CLEAN_DIR / "equity_returns_monthly.parquet",

    # market
    "market_prices_daily": RAW_DIR / "market_prices_daily.parquet",
    "market_returns_monthly": CLEAN_DIR / "market_returns_monthly.parquet",

    # rates / cds / fundamentals / etc (agregar cuando lleguemos)
    # "rates_monthly": CLEAN_DIR / "rates_monthly.parquet",
    # "cds_5y_monthly": CLEAN_DIR / "cds_5y_monthly.parquet",
    # "fundamentals_monthly": CLEAN_DIR / "fundamentals_monthly.parquet",

    # master
    "panel_master": CLEAN_DIR / "panel_master.parquet",
    "data_dictionary": CLEAN_DIR / "data_dictionary.csv",
    "qa_report": QA_DIR / "qa_report.csv",
}

# ---- Credenciales Refinitiv (NO hardcodear)
EIKON_APP_KEY = os.environ.get("EIKON_APP_KEY")
assert EIKON_APP_KEY, (
    "Falta la variable de entorno EIKON_APP_KEY. "
    "Ejemplo (Windows PowerShell): $env:EIKON_APP_KEY='TU_KEY'"
)

# ---- Parámetros operativos (conservadores)
REQUEST_PAUSE_SEC = 0.25          # pausa estándar entre requests
RETRY_MAX = 5                     # reintentos ante fallos transitorios
RETRY_BACKOFF_SEC = 2.0           # backoff base (se escala)

## 4) Logging y algunos helpers

In [ ]:
# ======================
# LOGGING + HELPERS
# ======================

def _build_logger(name: str = "etl") -> logging.Logger:
    logger = logging.getLogger(name)
    if logger.handlers:
        return logger  # evita duplicar handlers si re-ejecutás la celda

    logger.setLevel(logging.INFO)

    fmt = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s", "%Y-%m-%d %H:%M:%S")

    sh = logging.StreamHandler()
    sh.setFormatter(fmt)
    logger.addHandler(sh)

    fh = logging.FileHandler(LOG_DIR / "etl.log", encoding="utf-8")
    fh.setFormatter(fmt)
    logger.addHandler(fh)

    return logger


logger = _build_logger("etl")


def ensure_datetime(df: pd.DataFrame, col: str) -> pd.DataFrame:
    """
    Fuerza `df[col]` a datetime (naive, sin tz). No altera otras columnas.
    """
    if col not in df.columns:
        raise KeyError(f"Columna '{col}' no existe en df.")
    out = df.copy()
    out[col] = pd.to_datetime(out[col], errors="coerce")
    # Si viniera con tz, lo baja a naive
    if getattr(out[col].dt, "tz", None) is not None:
        out[col] = out[col].dt.tz_localize(None)
    return out


def safe_write_parquet(df: pd.DataFrame, path: Path, index: bool = False) -> None:
    """
    Escribe parquet y loguea tamaño y destino.
    """
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(path, index=index)
    logger.info(f"Output -> {path.as_posix()} | rows={len(df):,} cols={df.shape[1]}")


def sleep_request(pause_sec: float = REQUEST_PAUSE_SEC) -> None:
    time.sleep(pause_sec)

## 5) Conexión a refinitiv

In [ ]:
# ======================
# REFINTIV CONNECTION
# ======================

ek.set_app_key(EIKON_APP_KEY)
logger.info("Refinitiv/Eikon: APP_KEY seteada.")


## 6) Bonos corporativos — consolidación de emisores y tickers (Workspace export)

**Input:** Excels exportados manualmente desde Refinitiv Workspace (LSEG) por sector, ubicados en `data/inputs/bonds_empresas/`.  
**Output:** `data/inputs/empresas_tickers_test.csv` (Issuer, Ticker), usado como universo de firmas.  
**Chequeos:** columnas requeridas presentes; filas totales por archivo; duplicados y valores faltantes.  
**Notas:** esta etapa no consulta la API. Se usa export manual por falta de permisos para bajar metadata completa vía Python.

In [ ]:
# ======================
# BONOS (Workspace export) -> Universo Issuer / Ticker
# ======================

REQUIRED_COLS = ["Issuer", "Ticker"]

# Archivos exportados desde Workspace, organizados por sector
BOND_FILES = {
    "ConsumerStaples_and_Discretionary.xlsx": "CS_CD",
    "Energy.xlsx": "Energy",
    "Financials.xlsx": "Financials",
    "Health_Care.xlsx": "HealthCare",
    "Industrials.xlsx": "Industrials",
    "Technology_and_comms.xlsx": "Tech_Comms",
    "Utilities_and_Real_Estate.xlsx": "Utilities_RealEstate",
}

def read_bond_export(path: Path, sector: str) -> pd.DataFrame:
    """
    Lee un Excel exportado desde Workspace y devuelve
    Issuer + Ticker con etiqueta de sector.
    """
    # Validación básica de existencia
    if not path.exists():
        raise FileNotFoundError(f"No existe el archivo: {path}")

    df = pd.read_excel(path)

    # Chequeo de columnas mínimas
    missing = [c for c in REQUIRED_COLS if c not in df.columns]
    if missing:
        raise ValueError(
            f"Faltan columnas {missing} en {path.name}. "
            f"Columnas disponibles: {list(df.columns)}"
        )

    # Nos quedamos solo con lo necesario
    out = df[REQUIRED_COLS].copy()

    # Etiqueta de trazabilidad (no afecta el output final)
    out["sector_source"] = sector

    # Limpieza mínima de strings
    out["Issuer"] = out["Issuer"].astype(str).str.strip()
    out["Ticker"] = out["Ticker"].astype(str).str.strip()

    return out


logger.info(f"Leyendo archivos desde: {BONDS_DIR.as_posix()}")

frames = []

for filename, sector in BOND_FILES.items():
    fpath = BONDS_DIR / filename

    # Lectura individual por archivo
    df_part = read_bond_export(fpath, sector)
    logger.info(f"{filename} | filas={len(df_part):,}")

    frames.append(df_part)

# Consolidación total
df_bonds_universe = pd.concat(frames, ignore_index=True)

logger.info(f"Total consolidado | filas={len(df_bonds_universe):,}")

# Universo final: solo Issuer y Ticker, sin duplicados
df_universe_simple = (
    df_bonds_universe[["Issuer", "Ticker"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

# Escritura del CSV que se usa como input del pipeline
UNIVERSE_FILE.parent.mkdir(parents=True, exist_ok=True)
df_universe_simple.to_csv(UNIVERSE_FILE, index=False)

logger.info(
    f"Archivo generado: {UNIVERSE_FILE.as_posix()} "
    f"| filas={len(df_universe_simple):,}"
)

## 7) Empresas — descarga de metadata (Refinitiv)

**Input:** `data/inputs/empresas_tickers_test.csv` (Issuer, Ticker).  
**Output:** `data/raw/company_metadata.parquet` (+ `.csv`).  
**Chequeos:** cobertura de columnas clave (nombre, ISIN, CUSIP), listado de tickers con metadata incompleta y reintentos con sufijos.  
**Notas:** la descarga se hace por batches para evitar rate limits; algunos tickers requieren sufijos de mercado (ej. `.O`, `.N`, `.K`).

In [ ]:
# ======================
# EMPRESAS -> metadata Refinitiv (batches + reintentos)
# ======================

# Campos que se piden a Refinitiv
FIELDS = [
    "TR.CommonName",
    "TR.PrimaryRIC",
    "TR.ISIN",
    "TR.CUSIP",
    "TR.PermID",
    "TR.CompanyMarketCap",
    "TR.GICSSector",
    "TR.GICSSectorName",
    "TR.GICSIndustryGroup",
    "TR.GICSIndustryGroupName",
    "TR.GICSIndustry",
    "TR.GICSIndustryName",
    "TR.GICSSubIndustry",
    "TR.GICSSubIndustryName",
    "TR.HeadquartersCountry",
]

# Columnas mínimas para considerar "completo" (según nombres que devuelve ek.get_data)
REQUIRED_META_COLS = ["Company Common Name", "ISIN", "CUSIP"]

# Mapa manual de casos puntuales
MANUAL_RIC_MAP = {
    "ABBV": "ABBV.K",
    "ORCL": "ORCL.K",
}

def chunked(items: list[str], n: int) -> Iterable[list[str]]:
    """Corta una lista en batches de tamaño n."""
    for i in range(0, len(items), n):
        yield items[i:i + n]


def normalize_str(s: pd.Series) -> pd.Series:
    """Normaliza strings: strip y deja NaN donde corresponde."""
    out = s.astype("string").str.strip()
    out = out.replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
    return out


def fetch_metadata_batch(instruments: list[str], fields: list[str]) -> pd.DataFrame:
    """Pide metadata a Refinitiv para un batch y devuelve un DataFrame (puede venir vacío)."""
    df, err = ek.get_data(instruments, fields=fields)

    if err is not None and len(err):
        # No corta la corrida: solo deja registro
        logger.warning(f"Refinitiv devolvió warnings/errores (muestra): {str(err[:2])}")

    if df is None or df.empty:
        return pd.DataFrame()

    # Estandariza nombre de clave
    df = df.rename(columns={"Instrument": "ric"}).copy()
    if "ric" in df.columns:
        df["ric"] = normalize_str(df["ric"])

    # Identificadores típicos (si están)
    for col in ["ISIN", "CUSIP"]:
        if col in df.columns:
            df[col] = normalize_str(df[col])

    return df


def is_row_incomplete(df: pd.DataFrame) -> pd.Series:
    """Marca filas con metadata incompleta en columnas clave."""
    bad = pd.Series(False, index=df.index)
    for col in REQUIRED_META_COLS:
        if col not in df.columns:
            logger.warning(f"Falta columna esperada en respuesta: '{col}'")
            continue
        bad |= df[col].isna() | (df[col].astype("string").str.strip() == "")
    return bad


def build_retry_candidates(raw_ric: str) -> list[str]:
    """Arma candidatos de RIC para reintento (mapa manual o sufijos estándar)."""
    base = str(raw_ric).split()[0].split(".")[0].strip()
    if base in MANUAL_RIC_MAP:
        return [MANUAL_RIC_MAP[base]]
    return [f"{base}.O", f"{base}.N", f"{base}.K"]


def fetch_one_complete(ric: str) -> Optional[pd.DataFrame]:
    """Pide metadata para 1 RIC y devuelve df si viene completo (si no, None)."""
    try:
        df = fetch_metadata_batch([ric], FIELDS)
        if df.empty:
            return None

        bad = is_row_incomplete(df)
        if bad.any():
            return None

        return df

    except Exception as e:
        logger.warning(f"Error pidiendo metadata para {ric}: {type(e).__name__}: {e}")
        return None


# ----------------------
# 1) Leer universo (Issuer/Ticker)
# ----------------------
df_universe = pd.read_csv(UNIVERSE_FILE, dtype=str)

# Nos aseguramos de tener las columnas mínimas
missing_cols = [c for c in ["Ticker", "Issuer"] if c not in df_universe.columns]
if missing_cols:
    raise ValueError(f"Faltan columnas {missing_cols} en {UNIVERSE_FILE.name}")

df_universe = df_universe.dropna(subset=["Ticker", "Issuer"]).reset_index(drop=True)

rics = df_universe["Ticker"].astype(str).str.strip().tolist()
logger.info(f"Universo cargado: {len(rics):,} tickers")

# ----------------------
# 2) Primera pasada: descarga por batches
# ----------------------
frames = []
batch_size = 25

for batch in chunked(rics, batch_size):
    try:
        df_batch = fetch_metadata_batch(batch, FIELDS)
        if not df_batch.empty:
            frames.append(df_batch)
    except Exception as e:
        logger.warning(f"Error en batch (muestra): {batch[:3]} | {type(e).__name__}: {e}")
    sleep_request(REQUEST_PAUSE_SEC)

meta = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
logger.info(f"Metadata descargada (1ra pasada): filas={len(meta):,}")

if meta.empty:
    raise RuntimeError("La descarga de metadata devolvió un DataFrame vacío. Revisar conexión/credenciales/campos.")

# ----------------------
# 3) Detectar incompletos y armar lista de reintentos
# ----------------------
bad_mask = is_row_incomplete(meta)

bad_rics = (
    meta.loc[bad_mask, "ric"]
    .dropna()
    .astype("string")
    .unique()
    .tolist()
)

logger.info(f"RICs con metadata incompleta (1ra pasada): {len(bad_rics):,}")

# ----------------------
# 4) Reintentos: sufijos .O/.N/.K (y mapa manual)
# ----------------------
retry_frames: list[pd.DataFrame] = []
still_bad: list[str] = []

for r in bad_rics:
    candidates = build_retry_candidates(r)
    logger.info(f"Reintento: {r} | candidatos={candidates}")

    ok = False
    for cand in candidates:
        df_ok = fetch_one_complete(cand)
        if df_ok is not None and not df_ok.empty:
            retry_frames.append(df_ok)
            logger.info(f"OK: {r} -> {cand}")
            ok = True
            break
        else:
            logger.info(f"Sin datos completos: {cand}")

    if not ok:
        still_bad.append(r)

logger.info(f"RICs que siguen incompletos: {len(still_bad):,}")

# ----------------------
# 5) Consolidar: buenos + reintentos exitosos
# ----------------------
meta_good = meta.loc[~bad_mask].copy()

if retry_frames:
    meta_retry = pd.concat(retry_frames, ignore_index=True)

    # Normalización final de ids para evitar problemas al exportar
    for col in ["ric", "ISIN", "CUSIP"]:
        if col in meta_retry.columns:
            meta_retry[col] = normalize_str(meta_retry[col])

    meta_final = pd.concat([meta_good, meta_retry], ignore_index=True)
else:
    meta_final = meta_good

for col in ["ric", "ISIN", "CUSIP"]:
    if col in meta_final.columns:
        meta_final[col] = meta_final[col].astype("string")

# Resumen rápido de QA
logger.info(f"Metadata final: filas={len(meta_final):,} | cols={meta_final.shape[1]}")
na_counts = meta_final.isna().sum().sort_values(ascending=False).head(10)
logger.info(f"NAs (top 10): {na_counts.to_dict()}")

if still_bad:
    # No frena el pipeline: deja registro
    logger.warning(f"Lista de RICs incompletos (muestra): {still_bad[:15]}")

# ----------------------
# 6) Export
# ----------------------
out_parquet = RAW_DIR / "company_metadata.parquet"
out_csv = RAW_DIR / "company_metadata.csv"

safe_write_parquet(meta_final, out_parquet, index=False)
meta_final.to_csv(out_csv, index=False)
logger.info(f"Output -> {out_csv.as_posix()}")

## 8) Equity — precios diarios (CLOSE, VOLUME)

**Input:** `data/inputs/empresas_tickers_test.csv` (Ticker).  
**Output:** `data/raw/equity_prices_daily.parquet` (+ `.csv`).  
**Chequeos:** conteo de RICs con datos vs sin datos; rango de fechas por RIC; listado de series que arrancan tarde.  
**Notas:** para algunos tickers se prueban sufijos `.N` y `.O`. Se aplica pausa entre requests para reducir riesgos de rate limit (HTTP 429).

In [ ]:
# ======================
# EQUITY -> precios diarios (CLOSE, VOLUME)
# ======================

# Universo de tickers (sale del CSV de inputs)
df_emp = pd.read_csv(UNIVERSE_FILE, dtype=str).dropna(subset=["Ticker"]).copy()
df_emp["Ticker"] = df_emp["Ticker"].astype(str).str.strip()

tickers_raw = df_emp["Ticker"].tolist()

logger.info(f"Empresas en CSV: {len(df_emp):,}")
logger.info(f"Tickers detectados: {len(tickers_raw):,}")

def ric_candidates(t: str) -> list[str]:
    """Arma candidatos de RIC para equity según convención de mercado."""
    # Si ya viene con sufijo (ej. AAPL.O), se usa tal cual
    if "." in t:
        return [t]
    # Si no tiene sufijo, se prueban .N y .O
    return [f"{t}.N", f"{t}.O"]


all_frames: list[pd.DataFrame] = []
success = 0
fail = 0

for t in tickers_raw:
    got_it = False

    # Probamos candidatos por ticker
    for ric in ric_candidates(t):
        try:
            ts = ek.get_timeseries(
                ric,
                fields=["CLOSE", "VOLUME"],
                start_date=START_DATE,
                end_date=END_DATE,
                interval="daily",
            )

            # Si trajo datos, normalizamos y guardamos
            if ts is not None and not ts.empty:
                ts = ts.reset_index().rename(columns={"Date": "date"})
                ts["ric"] = ric

                ts["date"] = pd.to_datetime(ts["date"], errors="coerce")
                if getattr(ts["date"].dt, "tz", None) is not None:
                    ts["date"] = ts["date"].dt.tz_localize(None)

                all_frames.append(ts[["date", "ric", "CLOSE", "VOLUME"]])

                dmin = ts["date"].min()
                dmax = ts["date"].max()
                logger.info(f"OK {ric:<10} | {dmin.date()} -> {dmax.date()}")

                success += 1
                got_it = True
                break

        except Exception as e:
            logger.warning(f"Error con {ric}: {type(e).__name__}: {e}")

        # Pausa para rate limits
        sleep_request(0.6)

    if not got_it:
        logger.warning(f"Sin datos para ticker: {t}")
        fail += 1

logger.info(f"RICs con datos: {success:,}")
logger.info(f"RICs sin datos: {fail:,}")

# Consolidación final
if all_frames:
    out = pd.concat(all_frames, ignore_index=True)
else:
    out = pd.DataFrame(columns=["date", "ric", "CLOSE", "VOLUME"])

out = out.sort_values(["ric", "date"]).reset_index(drop=True)

# Export (usa nombres definidos en Config si los tenés ahí)
out_parquet = OUT.get("equity_prices_daily", RAW_DIR / "equity_prices_daily.parquet")
out_csv = out_parquet.with_suffix(".csv")

safe_write_parquet(out, out_parquet, index=False)
out.to_csv(out_csv, index=False)
logger.info(f"Output -> {out_csv.as_posix()}")
logger.info(f"Total filas: {len(out):,}")

# Chequeo: series que arrancan tarde
late = out.groupby("ric")["date"].min().reset_index()
late = late[late["date"] > pd.Timestamp("2016-01-01")]

if not late.empty:
    logger.warning("Series que arrancan tarde (min date > 2016-01-01):")
    logger.warning(late.sort_values("date").head(50).to_string(index=False))
else:
    logger.info("Cobertura temporal OK (no hay series que arranquen tarde según el umbral).")

## 9) Equity — retornos logarítmicos (diarios)

**Input:** `data/raw/equity_prices_daily.parquet` (date, ric, CLOSE, VOLUME).  
**Output:** `data/clean/equity_returns_daily.parquet` (+ `.csv`).  
**Chequeos:** filas de entrada, cantidad de RICs, filas con retorno válido, cantidad de RICs con retornos.  
**Notas:** el retorno se calcula como `ln(P_t) - ln(P_{t-1})` por activo.

In [ ]:
# ======================
# EQUITY -> retornos log diarios
# ======================

# Si venís del bloque anterior, `out` está en memoria.
# Si preferís hacerlo independiente, descomentá esta lectura.
# out = pd.read_parquet(OUT.get("equity_prices_daily", RAW_DIR / "equity_prices_daily.parquet"))

# Orden + tipo numérico
out = out.sort_values(["ric", "date"]).reset_index(drop=True).copy()
out["CLOSE"] = pd.to_numeric(out["CLOSE"], errors="coerce")

logger.info(f"Filas en precios (entrada): {len(out):,}")
logger.info(f"RICs únicos en precios: {out['ric'].nunique():,}")

# Retorno log: ln(P_t) - ln(P_{t-1}) por ric
out["ret_log"] = (
    out.groupby("ric")["CLOSE"]
       .transform(lambda s: np.log(s).diff())
)

# Se eliminan las primeras observaciones sin retorno (NaN por diff)
out_ret = out.dropna(subset=["ret_log"]).copy()

logger.info(f"Filas con ret_log válido: {len(out_ret):,}")
logger.info(f"RICs con al menos un retorno: {out_ret['ric'].nunique():,}")

# Export
out_parquet = OUT.get("equity_returns_daily", CLEAN_DIR / "equity_returns_daily.parquet")
out_csv = out_parquet.with_suffix(".csv")

safe_write_parquet(out_ret, out_parquet, index=False)
out_ret.to_csv(out_csv, index=False)
logger.info(f"Output -> {out_csv.as_posix()}")

## 10) Mercado (S&P 500) — serie diaria y merge con retornos de equity

**Input:** candidatos de mercado (TR / índice / ETFs) vía Refinitiv; `data/clean/equity_returns_daily.parquet` (date, ric, ret_log).  
**Output:** `data/raw/market_sp500_daily.parquet` y `data/clean/equity_market_returns_daily.parquet`.  
**Chequeos:** fuente elegida, rango de fechas, filas del merge y cobertura por fecha.  
**Notas:** se prueban varias fuentes por disponibilidad/permisos; se calcula retorno log diario del mercado.

In [ ]:
# ======================
# MERCADO S&P 500 -> serie diaria + merge con retornos de equity
# ======================

# Candidatos de mercado (orden de preferencia)
MARKET_CANDIDATES = [".SPXT", ".SPX", "SPY.P", "IVV.P", "VOO.P"]

def fetch_market_series(candidates: list[str]) -> tuple[str, pd.DataFrame]:
    """Intenta descargar una serie diaria de mercado y devuelve (ric_elegido, df)."""
    last_err: Optional[Exception] = None

    for ric in candidates:
        try:
            ts = ek.get_timeseries(
                ric,
                fields=["CLOSE"],
                start_date=START_DATE,
                end_date=END_DATE,
                interval="daily",
            )

            if ts is not None and not ts.empty:
                logger.info(f"Fuente de mercado elegida: {ric}")
                return ric, ts

        except Exception as e:
            last_err = e
            logger.warning(f"Intento fallido con {ric}: {type(e).__name__}: {e}")

        # Pausa suave entre intentos
        sleep_request(REQUEST_PAUSE_SEC)

    raise RuntimeError(
        "No se pudo descargar ninguna serie de mercado. "
        f"Último error: {type(last_err).__name__}: {last_err}" if last_err else
        "No se pudo descargar ninguna serie de mercado."
    )


chosen_ric, mkt_ts = fetch_market_series(MARKET_CANDIDATES)

# Normalización + retorno log del mercado
mkt = (
    mkt_ts.rename_axis(index="date")
          .reset_index()
          .rename(columns={"CLOSE": "mkt_close"})
)

mkt["date"] = pd.to_datetime(mkt["date"], errors="coerce")
if getattr(mkt["date"].dt, "tz", None) is not None:
    mkt["date"] = mkt["date"].dt.tz_localize(None)

mkt = mkt.sort_values("date").reset_index(drop=True)
mkt["mkt_close"] = pd.to_numeric(mkt["mkt_close"], errors="coerce")
mkt["mkt_ret_log"] = np.log(mkt["mkt_close"]).diff()

# Guardado de mercado
mkt_parquet = OUT.get("market_prices_daily", RAW_DIR / "market_sp500_daily.parquet")
mkt_csv = mkt_parquet.with_suffix(".csv")

safe_write_parquet(mkt, mkt_parquet, index=False)
mkt.to_csv(mkt_csv, index=False)
logger.info(f"Output -> {mkt_csv.as_posix()}")

# Merge equity + mercado (retornos)
eq_ret_path = OUT.get("equity_returns_daily", CLEAN_DIR / "equity_returns_daily.parquet")
eq_rets = pd.read_parquet(eq_ret_path)[["date", "ric", "ret_log"]].copy()

eq_rets["date"] = pd.to_datetime(eq_rets["date"], errors="coerce")
if getattr(eq_rets["date"].dt, "tz", None) is not None:
    eq_rets["date"] = eq_rets["date"].dt.tz_localize(None)

eq_mkt = (
    eq_rets.merge(mkt[["date", "mkt_ret_log"]], on="date", how="inner")
           .dropna(subset=["ret_log", "mkt_ret_log"])
           .sort_values(["ric", "date"])
           .reset_index(drop=True)
)

logger.info(f"Merge equity+mercado | filas={len(eq_mkt):,} | rics={eq_mkt['ric'].nunique():,}")

eq_mkt_parquet = OUT.get("equity_market_returns_daily", CLEAN_DIR / "equity_market_returns_daily.parquet")
safe_write_parquet(eq_mkt, eq_mkt_parquet, index=False)

## 11) ETFs sectoriales — precios diarios (CLOSE)

**Input:** lista de ETFs sectoriales (RICs) vía Refinitiv.  
**Output:** `data/raw/sector_etfs_daily.parquet` (+ `.csv`).  
**Chequeos:** ETFs con datos vs sin datos, rango de fechas por ETF, cantidad de filas por serie.  
**Notas:** se aplican reintentos con backoff ante rate limits (HTTP 429) y pausas entre requests.

In [ ]:
# ======================
# ETFs sectoriales -> precios diarios (CLOSE)
# ======================

# Lista de ETFs sectoriales (usar RICs si ya los tenés definidos así)
SECTOR_ETFS = [
    "XLF",     # financials
    "XLE",     # energy
    "XLK",     # tech
    "XLY",     # consumer discretionary
    "XLP",     # consumer staples
    "XLI",     # industrials
    "XLV",     # health care
    "XLU",     # utilities
    "XLRE.K",  # real estate
    "XLB",     # materials
    "XLC",     # communication services
]

def get_ts_safe(
    ric: str,
    start_date: str,
    end_date: str,
    fields: list[str] | None = None,
    max_attempts: int = 3,
) -> Optional[pd.DataFrame]:
    """
    Descarga una timeseries diaria con reintentos si aparece rate limit (429).
    Devuelve DataFrame o None si no se pudo.
    """
    if fields is None:
        fields = ["CLOSE"]

    for attempt in range(1, max_attempts + 1):
        try:
            ts = ek.get_timeseries(
                ric,
                fields=fields,
                start_date=start_date,
                end_date=end_date,
                interval="daily",
            )
            return ts

        except Exception as e:
            msg = str(e)

            # Si es 429, hacemos backoff y reintentamos
            if "429" in msg or "Too many requests" in msg:
                wait_sec = int(RETRY_BACKOFF_SEC * attempt * 2)
                logger.warning(f"429 para {ric} | espero {wait_sec}s | intento {attempt}/{max_attempts}")
                time.sleep(wait_sec)
                continue

            # Otros errores: se registra y se corta
            logger.warning(f"Error con {ric}: {type(e).__name__}: {e}")
            return None

    return None


sec_frames: list[pd.DataFrame] = []
missing: list[str] = []

logger.info(f"Descarga ETFs sectoriales | total={len(SECTOR_ETFS)}")

for ric in SECTOR_ETFS:
    logger.info(f"ETF sector: {ric}")

    ts = get_ts_safe(ric, START_DATE, END_DATE, fields=["CLOSE"], max_attempts=RETRY_MAX)

    if ts is None or ts.empty:
        logger.warning(f"Sin datos: {ric}")
        missing.append(ric)
        sleep_request(0.5)
        continue

    # Normalización estándar
    df = ts.reset_index().rename(columns={"Date": "date", "CLOSE": "price"})
    df["sector_ric"] = ric

    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    if getattr(df["date"].dt, "tz", None) is not None:
        df["date"] = df["date"].dt.tz_localize(None)

    df["price"] = pd.to_numeric(df["price"], errors="coerce")

    # Log corto con rango
    dmin = df["date"].min()
    dmax = df["date"].max()
    logger.info(f"OK {ric:<6} | {dmin.date()} -> {dmax.date()} | filas={len(df):,}")

    sec_frames.append(df)

    # Pausa corta para no saturar
    sleep_request(0.5)

if not sec_frames:
    raise RuntimeError("No se pudo descargar ningún ETF sectorial (todas las series vinieron vacías).")

sectors_ts = pd.concat(sec_frames, ignore_index=True)

# Guardado
out_parquet = OUT.get("sector_etfs_daily", RAW_DIR / "sector_etfs_daily.parquet")
out_csv = out_parquet.with_suffix(".csv")

safe_write_parquet(sectors_ts, out_parquet, index=False)
sectors_ts.to_csv(out_csv, index=False)
logger.info(f"Output -> {out_csv.as_posix()}")

# Resumen de cobertura
logger.info(f"ETFs con datos: {sectors_ts['sector_ric'].nunique():,}")
logger.info(f"Filas totales: {len(sectors_ts):,}")

if missing:
    logger.warning(f"ETFs sin datos (muestra): {missing[:10]}")

# Tabla rápida de rangos por ETF (útil para QA)
summary = sectors_ts.groupby("sector_ric")["date"].agg(["min", "max", "count"]).sort_values("count", ascending=False)
logger.info("Cobertura por ETF (top):")
logger.info(summary.head(12).to_string())

## 12) Fundamentals — descarga trimestral (Refinitiv)

**Input:** `data/inputs/empresas_tickers_test.csv` (Ticker) + overrides de RIC puntuales.  
**Output:** `data/raw/fundamentals_extended_q.parquet` (+ `.csv`).  
**Chequeos:** RICs con fundamentals vs sin fundamentals; quarters esperados por firma; duplicados por (ric, date).  
**Notas:** se descarga por batches con backoff ante rate limits (HTTP 429). La fecha trimestral se asigna por grilla Q 2015–2025 para uniformizar el panel.

In [ ]:
# ======================
# FUNDAMENTALS -> descarga trimestral (Q) por batches
# ======================

# Overrides puntuales de RIC para consultas (casos conocidos)
RIC_OVERRIDE = {
    "PEP":   "PEP.O",
    "ABBV":  "ABBV.K",
    "HON":   "HON.O",
    "AAPL":  "AAPL.O",
    "CSCO":  "CSCO.O",
    "ORCL":  "ORCL.K",
    "META":  "META.O",
    "GOOGL": "GOOGL.O",
    "MSFT":  "MSFT.O",
}

# Fields (fundamentals extendidos)
FUND_FIELDS = [
    "TR.F.TotAssets",
    "TR.F.TotLiab",
    "TR.F.TotShHoldEq",
    "TR.F.TotCurrAssets",
    "TR.F.TotCurrLiab",
    "TR.TotalDebtActValue",
    "TR.F.CashSTInvst",
    "TR.F.STDebtCurrPortOfLTDebt",
    "TR.F.DebtLTTot",
    "TR.F.TotRevenue",
    "TR.F.EBITDA",
    "TR.F.NetCashFlowOp",
    "TR.IntExp",
    "TR.F.CAPEXTot",
]

FUND_PARAMS = {
    "SDate": "2015-01-01",
    "EDate": "2025-12-31",
    "Frq": "Q",
    "Curn": "USD",
}

BATCH_SIZE = 20
SLEEP_BATCH_SEC = 1.0
MAX_RETRY = 3


def chunked(items: list[str], n: int) -> Iterable[list[str]]:
    """Corta lista en batches."""
    for i in range(0, len(items), n):
        yield items[i:i + n]


def get_data_safe(rics_batch: list[str], fields: list[str], params: dict, max_retry: int = 3) -> pd.DataFrame:
    """
    Wrapper para ek.get_data con reintentos ante 429.
    Devuelve DataFrame (puede venir vacío).
    """
    for attempt in range(1, max_retry + 1):
        try:
            df, err = ek.get_data(rics_batch, fields=fields, parameters=params)
            if err is not None and len(err):
                logger.warning(f"Refinitiv devolvió warnings/errores (muestra): {str(err[:2])}")
            return df if df is not None else pd.DataFrame()

        except Exception as e:
            msg = str(e)

            if "429" in msg or "Too many requests" in msg:
                wait_sec = 4 + (attempt - 1) * 4
                logger.warning(f"429 en batch | espero {wait_sec}s | intento {attempt}/{max_retry}")
                time.sleep(wait_sec)
                continue

            logger.warning(f"Error en batch (no 429): {type(e).__name__}: {e}")
            return pd.DataFrame()

    return pd.DataFrame()


def to_snake(col: str) -> str:
    """Pasa nombres tipo TR.F.X a snake_case."""
    x = col.replace("TR.", "").replace("F.", "").replace(":", ".")
    x = x.replace(".", "_")
    return x.lower()


# ----------------------
# 1) Universo + ric_query
# ----------------------
df_emp = pd.read_csv(UNIVERSE_FILE, dtype=str).dropna(subset=["Ticker"]).reset_index(drop=True)

df_emp["ticker"] = df_emp["Ticker"].astype(str).str.strip()
df_emp["ric_query"] = df_emp["ticker"].map(RIC_OVERRIDE).fillna(df_emp["ticker"])

rics = df_emp["ric_query"].dropna().astype(str).unique().tolist()

logger.info(f"Empresas en universo: {len(df_emp):,}")
logger.info(f"RICs únicos (consulta): {len(rics):,}")

# ----------------------
# 2) Descarga por batches
# ----------------------
all_frames: list[pd.DataFrame] = []

logger.info("Descargando fundamentals (Q) por batches...")

for i, batch in enumerate(chunked(rics, BATCH_SIZE), start=1):
    logger.info(f"Batch {i} | size={len(batch)}")

    df_batch = get_data_safe(batch, FUND_FIELDS, FUND_PARAMS, max_retry=MAX_RETRY)
    if not df_batch.empty:
        all_frames.append(df_batch)

    time.sleep(SLEEP_BATCH_SEC)

if not all_frames:
    raise RuntimeError("No se descargaron fundamentals. Revisar fields/permisos/conexión.")

fund = pd.concat(all_frames, ignore_index=True)

# Clave de instrumento
if "Instrument" in fund.columns:
    fund = fund.rename(columns={"Instrument": "ric"})

# Snake_case (incluye los fields y las columnas extra que traiga Refinitiv)
fund.columns = [to_snake(c) for c in fund.columns]

# ----------------------
# 3) Mapeo de rastreo (ticker original)
# ----------------------
df_map = df_emp[["ticker", "ric_query"]].rename(columns={"ric_query": "ric"})
fund = fund.merge(df_map, on="ric", how="left")

# ----------------------
# 4) Fecha trimestral (grilla Q 2015–2025)
# ----------------------
start_q = pd.Timestamp("2015-01-01")
end_q   = pd.Timestamp("2025-12-31")
full_q_range = pd.date_range(start=start_q, end=end_q, freq="Q")

def assign_quarter_grid(g: pd.DataFrame) -> pd.DataFrame:
    # Orden estable dentro de cada ric
    g = g.reset_index(drop=True)
    n = len(g)

    reps = math.ceil(n / len(full_q_range))
    dates = list(full_q_range) * reps
    g["date"] = dates[:n]
    return g

fund = fund.groupby("ric", group_keys=False).apply(assign_quarter_grid)

# Dedupe por (ric, date) si quedó repetido
fund = (
    fund.sort_values(["ric", "date"])
        .groupby(["ric", "date"], as_index=False)
        .first()
)

# ----------------------
# 5) QA y export
# ----------------------
logger.info(f"Fundamentals final | filas={len(fund):,} | rics={fund['ric'].nunique():,}")
logger.info(f"Quarters esperados (grilla): {len(full_q_range):,}")

missing = df_emp.loc[~df_emp["ric_query"].isin(fund["ric"]), ["Issuer", "ticker", "ric_query"]]
if not missing.empty:
    logger.warning(f"Empresas sin fundamentals: {len(missing):,}")
    logger.warning(missing.head(20).to_string(index=False))

out_parquet = OUT.get("fundamentals_q", RAW_DIR / "fundamentals_extended_q.parquet")
out_csv = out_parquet.with_suffix(".csv")

safe_write_parquet(fund, out_parquet, index=False)
fund.to_csv(out_csv, index=False)
logger.info(f"Output -> {out_csv.as_posix()}")

## 13) Curva soberana USD (UST) — yields diarios por tenor

**Input:** RICs de UST (`US1MT=RR`, `US3MT=RR`, …) vía Refinitiv.  
**Output:** `data/raw/ust_yield_curve_daily.parquet` (+ `.csv`).  
**Chequeos:** tenores con datos vs sin datos, rango de fechas por tenor, cantidad de filas.  
**Notas:** se aplican reintentos con backoff ante rate limits (HTTP 429) y pausas entre requests.

In [ ]:
# ======================
# UST yield curve -> daily (por tenor)
# ======================

UST_RICS = {
    "US1M":  "US1MT=RR",
    "US3M":  "US3MT=RR",
    "US6M":  "US6MT=RR",
    "US1Y":  "US1YT=RR",
    "US2Y":  "US2YT=RR",
    "US3Y":  "US3YT=RR",
    "US5Y":  "US5YT=RR",
    "US7Y":  "US7YT=RR",
    "US10Y": "US10YT=RR",
    "US30Y": "US30YT=RR",
}

def get_ts_safe(
    ric: str,
    start_date: str,
    end_date: str,
    fields: list[str] | None = None,
    max_attempts: int = 3,
) -> Optional[pd.DataFrame]:
    """Timeseries diaria con reintentos si cae 429. Devuelve DataFrame o None."""
    if fields is None:
        fields = ["CLOSE"]

    for attempt in range(1, max_attempts + 1):
        try:
            ts = ek.get_timeseries(
                ric,
                fields=fields,
                start_date=start_date,
                end_date=end_date,
                interval="daily",
            )
            return ts

        except Exception as e:
            msg = str(e)

            if "429" in msg or "Too many requests" in msg:
                wait_sec = 4 + (attempt - 1) * 4
                logger.warning(f"429 para {ric} | espero {wait_sec}s | intento {attempt}/{max_attempts}")
                time.sleep(wait_sec)
                continue

            logger.warning(f"Error con {ric}: {type(e).__name__}: {e}")
            return None

    return None


logger.info("Descargando curva UST (USD) por tenor...")
logger.info(f"Tenores: {list(UST_RICS.keys())}")

ust_frames: list[pd.DataFrame] = []
missing: list[str] = []

for tenor, ric in UST_RICS.items():
    logger.info(f"UST {tenor} | {ric}")

    ts = get_ts_safe(ric, START_DATE, END_DATE, fields=["CLOSE"], max_attempts=RETRY_MAX)

    if ts is None or ts.empty:
        logger.warning(f"Sin datos: {tenor} ({ric})")
        missing.append(tenor)
        sleep_request(0.4)
        continue

    df = ts.reset_index().rename(columns={"Date": "date", "CLOSE": "yield"})
    df["tenor"] = tenor
    df["ric"] = ric

    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    if getattr(df["date"].dt, "tz", None) is not None:
        df["date"] = df["date"].dt.tz_localize(None)

    df["yield"] = pd.to_numeric(df["yield"], errors="coerce")

    dmin = df["date"].min()
    dmax = df["date"].max()
    logger.info(f"OK {tenor:<4} | {dmin.date()} -> {dmax.date()} | filas={len(df):,}")

    ust_frames.append(df)

    # Pausa corta para no saturar
    sleep_request(0.4)

if not ust_frames:
    raise RuntimeError("No se pudo descargar ningún punto de la curva UST.")

ust = pd.concat(ust_frames, ignore_index=True)

# Export
out_parquet = OUT.get("ust_yield_curve_daily", RAW_DIR / "ust_yield_curve_daily.parquet")
out_csv = out_parquet.with_suffix(".csv")

safe_write_parquet(ust, out_parquet, index=False)
ust.to_csv(out_csv, index=False)
logger.info(f"Output -> {out_csv.as_posix()}")

# Resumen corto
logger.info(f"Tenores con datos: {ust['tenor'].nunique():,}")
logger.info(f"Filas totales: {len(ust):,}")
logger.info(f"Rango global: {ust['date'].min().date()} -> {ust['date'].max().date()}")

if missing:
    logger.warning(f"Tenores sin datos: {missing}")

## 14) CDS corporativo — spreads diarios (fallback 5Y → genérico)

**Input:** `data/inputs/empresas_tickers_test.csv` (Ticker/Issuer).  
**Output:** `data/raw/cds_spreads_daily.parquet` (+ `.csv`).  
**Chequeos:** emisores con CDS vs sin CDS, campo usado (5Y o genérico), rango de fechas y filas totales.  
**Notas:** se intenta primero `TR.CDSMidSpread5Y`; si no hay cobertura, se usa `TR.CDSMidSpread`. Se aplican pausas y reintentos ante 429.

In [ ]:
# ======================
# CDS -> spreads diarios (5Y con fallback)
# ======================

# Fechas salen del Config general (START_DATE / END_DATE)

# Pausas operativas
SLEEP_EACH_SEC = 0.6
BATCH_SIZE = 25
SLEEP_BATCH_SEC = 8

def get_cds_ts_safe(
    ric: str,
    start_date: str,
    end_date: str,
    max_attempts: int = 3,
) -> tuple[Optional[pd.DataFrame], Optional[str]]:
    """
    Baja CDS diario:
    - intenta TR.CDSMidSpread5Y
    - si viene vacío, cae a TR.CDSMidSpread
    Reintenta ante 429. Devuelve (df, field_used) o (None, None).
    """
    for attempt in range(1, max_attempts + 1):
        try:
            # 1) CDS 5Y
            field_used = "TR.CDSMidSpread5Y"
            ts = ek.get_timeseries(
                ric,
                fields=[field_used],
                start_date=start_date,
                end_date=end_date,
                interval="daily",
            )

            # 2) Fallback genérico
            if ts is None or ts.empty:
                field_used = "TR.CDSMidSpread"
                ts = ek.get_timeseries(
                    ric,
                    fields=[field_used],
                    start_date=start_date,
                    end_date=end_date,
                    interval="daily",
                )

            if ts is None or ts.empty:
                return None, None

            return ts, field_used

        except Exception as e:
            msg = str(e)

            if "429" in msg or "Too many requests" in msg:
                wait_sec = 4 + (attempt - 1) * 4
                logger.warning(f"429 con {ric} | espero {wait_sec}s | intento {attempt}/{max_attempts}")
                time.sleep(wait_sec)
                continue

            logger.warning(f"Error con {ric}: {type(e).__name__}: {e}")
            return None, None

    return None, None


# ----------------------
# 1) Universo de emisores
# ----------------------
df_emp = pd.read_csv(UNIVERSE_FILE, dtype=str).dropna(subset=["Ticker"]).copy()
df_emp["ric"] = df_emp["Ticker"].astype(str).str.strip().replace({"": pd.NA})
df_emp = df_emp.dropna(subset=["ric"]).reset_index(drop=True)

rics = df_emp["ric"].unique().tolist()

logger.info(f"Emisores a pedir CDS: {len(rics):,}")

# ----------------------
# 2) Descarga CDS
# ----------------------
frames: list[pd.DataFrame] = []
ok_count = 0
no_cds_count = 0

for i, ric in enumerate(rics, start=1):
    ts, field_used = get_cds_ts_safe(ric, START_DATE, END_DATE, max_attempts=RETRY_MAX)

    if ts is not None and not ts.empty and field_used is not None:
        df = ts.reset_index().rename(columns={"Date": "date", field_used: "cds_spread_bps"})
        df["ric"] = ric
        df["field_used"] = field_used  # trazabilidad

        df["date"] = pd.to_datetime(df["date"], errors="coerce")
        if getattr(df["date"].dt, "tz", None) is not None:
            df["date"] = df["date"].dt.tz_localize(None)

        df["cds_spread_bps"] = pd.to_numeric(df["cds_spread_bps"], errors="coerce")

        frames.append(df)
        ok_count += 1

        dmin = df["date"].min()
        dmax = df["date"].max()
        logger.info(f"OK {ric:<8} | {field_used} | {dmin.date()} -> {dmax.date()} | filas={len(df):,}")

    else:
        no_cds_count += 1
        logger.info(f"Sin CDS: {ric}")

    # Pausa corta por emisor
    sleep_request(SLEEP_EACH_SEC)

    # Pausa larga cada BATCH_SIZE emisores
    if i % BATCH_SIZE == 0:
        logger.info(f"Pausa batch ({i} emisores) | {SLEEP_BATCH_SEC}s")
        time.sleep(SLEEP_BATCH_SEC)

logger.info(f"Emisores con CDS: {ok_count:,}")
logger.info(f"Emisores sin CDS: {no_cds_count:,}")
logger.info(f"Total emisores: {len(rics):,}")

# ----------------------
# 3) Consolidación + export
# ----------------------
if not frames:
    raise RuntimeError("Ningún emisor devolvió CDS. Revisar cobertura/permisos/campos.")

cds_all = pd.concat(frames, ignore_index=True)

# Enriquecemos con Issuer/Ticker si está en el CSV
meta_small = df_emp.rename(columns={"Ticker": "Ticker_raw"})
cds_all = cds_all.merge(meta_small[["ric", "Issuer", "Ticker_raw"]], on="ric", how="left")

cds_all = cds_all.sort_values(["ric", "date"]).reset_index(drop=True)

out_parquet = OUT.get("cds_spreads_daily", RAW_DIR / "cds_spreads_daily.parquet")
out_csv = out_parquet.with_suffix(".csv")

safe_write_parquet(cds_all, out_parquet, index=False)
cds_all.to_csv(out_csv, index=False)
logger.info(f"Output -> {out_csv.as_posix()}")

logger.info(f"Filas CDS: {len(cds_all):,}")
logger.info(f"Rango global: {cds_all['date'].min().date()} -> {cds_all['date'].max().date()}")

## 15) Revenue + market share (SP500) — revenue trimestral y participación por industria

**Input:** universo `0#.SPX` vía Refinitiv + `data/inputs/empresas_tickers_test.csv` para identificar la muestra.  
**Output:**  
- `data/raw/gics_industry_group_sales_quarterly_sp500.csv` (ventas agregadas por industria y trimestre)  
- `data/raw/fundamentals_with_market_share_sp500.csv` (panel SP500)  
- `data/raw/fundamentals_with_market_share.csv` (panel filtrado a la muestra)  

**Chequeos:** fuente de revenue usada, cobertura de GICS, filas por trimestre, market share en [0,1] (cuando hay industria).  
**Notas:** revenue se toma de un set de campos candidatos según disponibilidad/permisos. La agrupación base es `GICS Industry Group` (se puede cambiar a sector/industry si hiciera falta).

In [ ]:
# ======================
# Revenue + market share (SP500)
# ======================

# ----------------------
# Helpers de descarga
# ----------------------
def chunked(items: list[str], n: int) -> Iterable[list[str]]:
    """Corta lista en batches."""
    for i in range(0, len(items), n):
        yield items[i:i + n]


def get_data_safe(
    rics_batch: list[str],
    fields: list[str],
    params: dict,
    max_retry: int = 3,
) -> pd.DataFrame:
    """
    Wrapper para ek.get_data con reintentos ante 429.
    Devuelve DataFrame (puede venir vacío).
    """
    for attempt in range(1, max_retry + 1):
        try:
            df, err = ek.get_data(rics_batch, fields=fields, parameters=params)
            if err is not None and len(err):
                logger.warning(f"Refinitiv warnings/errores (muestra): {str(err[:2])}")
            return df if df is not None else pd.DataFrame()

        except Exception as e:
            msg = str(e)
            if "429" in msg or "Too many requests" in msg:
                wait_sec = 4 + (attempt - 1) * 4
                logger.warning(f"429 en batch | espero {wait_sec}s | intento {attempt}/{max_retry}")
                time.sleep(wait_sec)
                continue

            logger.warning(f"Error en batch (no 429): {type(e).__name__}: {e}")
            return pd.DataFrame()

    return pd.DataFrame()


def get_metadata_batched(
    instruments: list[str],
    fields: list[str],
    batch_size: int = 25,
    pause_sec: float = 0.4,
) -> pd.DataFrame:
    """Baja metadata por batches usando ek.get_data (sin parameters)."""
    frames: list[pd.DataFrame] = []

    for i, batch in enumerate(chunked(instruments, batch_size), start=1):
        try:
            df_batch, err = ek.get_data(batch, fields=fields)
            if err is not None and len(err):
                logger.warning(f"Warnings/errores metadata batch (muestra): {str(err[:2])}")
            if df_batch is not None and not df_batch.empty:
                frames.append(df_batch)
        except Exception as e:
            logger.warning(f"Error metadata batch {i} | {type(e).__name__}: {e}")

        time.sleep(pause_sec)

    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


def first_present(cands: list[str], cols_set: set[str]) -> Optional[str]:
    """Devuelve el primer candidato que exista en cols_set."""
    for c in cands:
        if c in cols_set:
            return c
    return None


# ----------------------
# 1) Muestra (para filtrar al final)
# ----------------------
df_emp = pd.read_csv(UNIVERSE_FILE, dtype=str)
df_emp["Ticker"] = df_emp["Ticker"].astype(str).str.strip()

RIC_FIX = {
    "PEP": "PEP.O",
    "ABBV": "ABBV.K",
    "HON": "HON.O",
    "AAPL": "AAPL.O",
    "CSCO": "CSCO.O",
    "ORCL": "ORCL.K",
    "META": "META.O",
    "GOOGL": "GOOGL.O",
    "MSFT": "MSFT.O",
}

def fix_ticker(t: str) -> str:
    t = str(t).strip()
    if not t:
        return t
    base = t.split(".")[0]
    return RIC_FIX.get(base, t)

df_emp["Ticker_fixed"] = df_emp["Ticker"].apply(fix_ticker)
df_emp["ric_base"] = df_emp["Ticker_fixed"].str.split(".").str[0]
target_bases = sorted(df_emp["ric_base"].dropna().unique().tolist())

logger.info(f"Muestra (bases RIC): {len(target_bases):,}")

# ----------------------
# 2) Universo SP500 (0#.SPX)
# ----------------------
logger.info("Descargando universo SP500 (0#.SPX) ...")
spx_df, spx_err = ek.get_data("0#.SPX", ["TR.RIC"])
if spx_err is not None and len(spx_err):
    logger.warning(f"Warnings universo SPX (muestra): {str(spx_err[:3])}")

if spx_df is None or spx_df.empty:
    raise RuntimeError("No llegó universo SPX desde Refinitiv.")

spx_df = spx_df.dropna(subset=["RIC"]).copy()
spx_df["RIC"] = spx_df["RIC"].astype(str).str.strip()
spx_rics = spx_df["RIC"].dropna().unique().tolist()

logger.info(f"RICs SP500 detectados: {len(spx_rics):,}")

# ----------------------
# 3) Metadata GICS SP500
# ----------------------
META_FIELDS = [
    "TR.CommonName",
    "TR.PrimaryRIC",
    "TR.ISIN",
    "TR.CUSIP",
    "TR.PermID",
    "TR.CompanyMarketCap",
    "TR.GICSSector",
    "TR.GICSSectorName",
    "TR.GICSIndustryGroup",
    "TR.GICSIndustryGroupName",
    "TR.GICSIndustry",
    "TR.GICSIndustryName",
    "TR.GICSSubIndustry",
    "TR.GICSSubIndustryName",
    "TR.HeadquartersCountry",
]

logger.info("Descargando metadata (GICS) para SP500 ...")
meta_spx = get_metadata_batched(spx_rics, META_FIELDS, batch_size=25, pause_sec=0.4)

if meta_spx.empty:
    raise RuntimeError("No llegó metadata SPX (GICS).")

meta_spx.columns = [str(c).strip() for c in meta_spx.columns]
meta_spx = meta_spx.rename(columns={
    "Instrument": "ric",
    "GICS Sector Name": "gics_sector_name",
    "GICS Industry Group Name": "gics_industry_group_name",
    "GICS Industry Name": "gics_industry_name",
    "GICS Sub-Industry Name": "gics_subindustry_name",
})

meta_spx["ric"] = meta_spx["ric"].astype(str).str.strip()

gics_cols = [
    "gics_sector_name",
    "gics_industry_group_name",
    "gics_industry_name",
    "gics_subindustry_name",
]

meta_spx_gics = meta_spx[["ric"] + gics_cols].drop_duplicates(subset=["ric"]).copy()
logger.info(f"Metadata GICS SP500 | filas={len(meta_spx_gics):,}")

# ----------------------
# 4) Fundamentals (revenue) SP500
# ----------------------
REV_CANDS = [
    "TR.F.TotRevenue",
    "TR.F.SalesTurnover",
    "TR.F.TotalRevenue",
    "TR.F.SalesOrRevenue",
    "TR.F.NetSales",
    "TR.F.Revenue",
    "TR.F.RevenueReported",
]

fields_funda = REV_CANDS + ["TR.F.PeriodEndDate"]

params = {
    "SDate": "2015-01-01",
    "EDate": "2025-12-31",
    "Frq": "Q",
    "Curn": "USD",
}

logger.info("Descargando fundamentals (revenue) para SP500 ...")

funda_frames: list[pd.DataFrame] = []
BATCH = 20
SLEEP_SEC = 1.0

for i, batch in enumerate(chunked(spx_rics, BATCH), start=1):
    logger.info(f"Batch funda {i} | size={len(batch)}")
    df_b = get_data_safe(batch, fields_funda, params, max_retry=RETRY_MAX)
    if df_b is not None and not df_b.empty:
        funda_frames.append(df_b)
    time.sleep(SLEEP_SEC)

if not funda_frames:
    raise RuntimeError("No llegó fundamentals de revenue.")

funda_all = pd.concat(funda_frames, ignore_index=True)
funda_all.columns = [str(c).strip() for c in funda_all.columns]
cols = set(funda_all.columns)

ric_col = first_present(["Instrument", "RIC", "TR.RIC", "Constituent RIC"], cols)
ped_col = first_present(["TR.F.PeriodEndDate", "Period End Date", "PeriodEndDate"], cols)
rev_col = first_present(
    REV_CANDS + ["Revenue from Business Activities - Total", "Revenue", "Sales/Turnover"],
    cols,
)

missing = [x for x, y in [("ric", ric_col), ("period_end", ped_col), ("revenue", rev_col)] if y is None]
if missing:
    raise RuntimeError(f"Columnas faltantes en fundamentals: {missing} | disponibles={sorted(cols)}")

logger.info(f"Campo revenue usado: {rev_col}")

funda_all = funda_all.rename(columns={ric_col: "ric", ped_col: "period_end", rev_col: "revenue"})
funda_all["ric"] = funda_all["ric"].astype(str).str.strip()
funda_all["period_end"] = pd.to_datetime(funda_all["period_end"], errors="coerce")
funda_all["revenue"] = pd.to_numeric(funda_all["revenue"], errors="coerce")

funda_all = funda_all.dropna(subset=["period_end"]).copy()
logger.info(f"Fundamentals revenue | filas={len(funda_all):,}")

# ----------------------
# 5) Merge + ventas industria + market share
# ----------------------
funda_all = funda_all.merge(meta_spx_gics, on="ric", how="left")

group_col = "gics_industry_group_name"
if group_col not in funda_all.columns:
    raise RuntimeError(f"No existe '{group_col}' en funda_all. Columnas={funda_all.columns.tolist()}")

sector_sales = (
    funda_all.dropna(subset=[group_col])
             .groupby([group_col, "period_end"], as_index=False)["revenue"]
             .sum()
             .rename(columns={"revenue": "industry_sales"})
)

out_sales = OUT.get(
    "gics_industry_group_sales_q_sp500",
    RAW_DIR / "gics_industry_group_sales_quarterly_sp500.csv"
)
sector_sales.to_csv(out_sales, index=False)
logger.info(f"Output -> {out_sales.as_posix()} | filas={len(sector_sales):,}")

funda_all = funda_all.merge(sector_sales, on=[group_col, "period_end"], how="left")
funda_all["market_share"] = funda_all["revenue"] / funda_all["industry_sales"]

# ----------------------
# 6) Export: SP500 completo + muestra
# ----------------------
funda_all["ric_base"] = funda_all["ric"].astype(str).str.split(".").str[0]

out_sp500 = OUT.get("fundamentals_with_market_share_sp500", RAW_DIR / "fundamentals_with_market_share_sp500.csv")
funda_all.to_csv(out_sp500, index=False)
logger.info(f"Output -> {out_sp500.as_posix()} | filas={len(funda_all):,}")

funda_focus = funda_all[funda_all["ric_base"].isin(target_bases)].copy()
out_focus = OUT.get("fundamentals_with_market_share_focus", RAW_DIR / "fundamentals_with_market_share.csv")
funda_focus.to_csv(out_focus, index=False)
logger.info(f"Output -> {out_focus.as_posix()} | filas={len(funda_focus):,}")

# QA rápido: market_share fuera de rango (solo donde hay industria_sales)
bad_ms = funda_all.loc[funda_all["industry_sales"].notna(), "market_share"]
share_oob = ((bad_ms < 0) | (bad_ms > 1)).sum()
if share_oob:
    logger.warning(f"Market share fuera de [0,1] (conteo): {int(share_oob)}")